In [3]:
import pandas as pd
import openpyxl
import re 

# ===========================
# 1. LOAD DEDUP FILE (BYPASS PANDAS BUG)
# ===========================

wb = openpyxl.load_workbook("downloads/RGB-T_Clean.xlsx")
ws = wb.active

data = list(ws.values)

# récupérer header + données
df = pd.DataFrame(data[1:], columns=data[0])

print("Records loaded:", len(df))


# ===========================
# 2. NORMALIZATION
# ===========================
def norm(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()


# ===========================
# 3. KEYWORD GROUPS (RGB-T)
# ===========================

# Strong RGB-T / fusion
MULTIMODAL_STRONG = [
    "rgb-t", "rgbt", "fusion", "multispectral", "infrared"
]

# Weak multimodal
MULTIMODAL_WEAK = [
    "thermal imaging", "cross-modal", "multi-modal"
]

# Tasks
TASK = [
    "object detection", "segmentation", "tracking",
    "pedestrian", "re-identification", "autonomous driving"
]

# AI/ML
AI = [
    "machine learning", "deep learning", "neural", "cnn",
    "transformer", "model", "algorithm"
]

# Exclusions
NON_ORIGINAL = [
    "review", "survey", "meta-analysis", "editorial", "commentary"
]

IRRELEVANT = [
    "medical", "disease", "health", "finance", "marketing"
]


# ===========================
# 4. HELPER
# ===========================
def contains_any(text, keywords):
    return any(k in text for k in keywords)


# ===========================
# 5. SCORING FUNCTION
# ===========================
def score_record(title, abstract):

    combined = norm(title) + " " + norm(abstract)
    t = " " + combined + " "

    # Hard exclusions
    if contains_any(t, NON_ORIGINAL):
        return "Exclude", "Non-original research", 0

    if contains_any(t, IRRELEVANT):
        return "Exclude", "Irrelevant domain", 0

    score = 0
    reasons = []

    # RGB-T / fusion
    if contains_any(t, MULTIMODAL_STRONG):
        score += 3
        reasons.append("RGB-T/fusion")

    elif contains_any(t, MULTIMODAL_WEAK):
        score += 1
        reasons.append("weak multimodal")

    # Task
    if contains_any(t, TASK):
        score += 2
        reasons.append("task")

    # AI
    if contains_any(t, AI):
        score += 1
        reasons.append("AI")

    # Decision
    if score >= 4:
        return "Include", "Score≥4: " + ", ".join(reasons), score

    if score >= 2:
        return "Maybe", "Score 2-3: " + ", ".join(reasons), score

    return "Exclude", "Low score: " + ", ".join(reasons), score


# ===========================
# 6. APPLY SCORING
# ===========================
out = df.copy()

if "Abstract" not in out.columns:
    out["Abstract"] = ""

out[["Decision", "Reason", "Score"]] = out.apply(
    lambda row: pd.Series(score_record(row["Title"], row["Abstract"])),
    axis=1
)


# ===========================
# 7. SUMMARY
# ===========================
print("\nDecision counts:")
print(out["Decision"].value_counts())


# ===========================
# 8. SAVE
# ===========================
out.to_excel("downloads/RGB-T_screened.xlsx", index=False)

print("\nSaved: downloads/RGB-T_screened.xlsx")

Records loaded: 135

Decision counts:
Include    96
Maybe      34
Exclude     5
Name: Decision, dtype: int64

Saved: downloads/RGB-T_screened.xlsx
